# 01 -- Legacy Cache Audit (diagnostic only)

Runs the **read-only, diagnostic-only** audit of the legacy `Arnold_Quant` repository's cache artifacts. This audit:

- **Never deserializes pickle.** Pickle artifacts are hashed and classified by metadata only.
- **Never modifies** anything under the legacy repository.
- Never treats a legacy artifact as `verified_compatible` merely because it exists -- every classification here is, at best, `partially_verified` or `diagnostic_only`.

Set `LEGACY_REPO_PATH` below to your own checkout. If you don't have one, skip straight to the Findings cell -- this notebook degrades gracefully.

In [ ]:
from pathlib import Path

LEGACY_REPO_PATH = Path.home() / "Downloads" / "Arnold_Quant"  # <-- set to your own checkout
print("legacy repo path:", LEGACY_REPO_PATH, "exists:", LEGACY_REPO_PATH.exists())


In [ ]:
from atlas_quant.strategies.multi_factor_ranking_ml.production.legacy_audit import audit_legacy_repository

if LEGACY_REPO_PATH.exists():
    results = audit_legacy_repository(LEGACY_REPO_PATH)
    for r in results:
        print(f"{r.relative_path:30s} exists={r.exists!s:5s} format={r.file_format:10s} "
              f"classification={r.classification.value:20s} {r.reason[:70]}")
else:
    results = ()
    print("no legacy repository at this path -- skipping (this is fine; the audit is optional)")


In [ ]:
# Confirms this notebook made no writes under the legacy repository.
if LEGACY_REPO_PATH.exists():
    import subprocess
    status = subprocess.run(
        ["git", "status", "--short"], capture_output=True, text=True, cwd=str(LEGACY_REPO_PATH),
    ).stdout
    print("git status --short in the legacy repo (should be unaffected by this notebook):")
    print(status or "(clean or unchanged by this notebook)")


## Findings

- Every pickle artifact is classified `diagnostic_only` from hash/metadata alone -- never deserialized.
- JSON artifacts are, at best, `partially_verified` (well-formed, but content-level compatibility with current AtlasQuant identities is not yet independently confirmed).

## Limitations

- A `diagnostic_only`/`partially_verified` classification is **not** authorization to reuse a legacy cache in a production run. Per this project's own constraint: legacy caches may only be reused after their provenance and configuration identity have been independently verified, and never merely because they produce performance close to the old report.